In [ ]:
client_id = "59v9on4emf"
client_secret = "894894550d"

In [ ]:
import httpx
import json

# --- 1. CONFIGURATION ---
# Change these values to match your specific setup
ERPNEXT_URL = "https://erpnext-ai-integration.m.frappe.cloud.com"  # e.g., https://mycompany.com
EMAIL = "gharadpatil75@gmail.com"
PASSWORD = "@Mahendra@123"

print(f"🔄 Attempting to connect to ERPNext instance at: {ERPNEXT_URL}...\n")

# Use an asynchronous or synchronous client. For Jupyter notebooks, 
# a synchronous client (httpx.Client) is easiest to run inline.
with httpx.Client(timeout=15.0) as client:
    
    # --- STEP 2: AUTHENTICATE & GET COOKIES ---
    login_url = f"{ERPNEXT_URL}/api/method/login"
    login_payload = {
        "usr": EMAIL,
        "pwd": PASSWORD
    }
    
    print("Step 1: Sending email and password to ERPNext...")
    login_response = client.post(login_url, data=login_payload)
    
    if login_response.status_code != 200 or login_response.json().get("message") != "Logged In":
        print("❌ Login Failed!")
        print(f"Status Code: {login_response.status_code}")
        print(f"Response Body: {login_response.text}")
        raise ValueError("Stop execution: Check your URL, Email, or Password.")
        
    print("✅ Successfully logged in via session cookies!")
    print(f"Returned Message: {login_response.json()}")
    print("-" * 50)

    # --- STEP 3: DYNAMICALLY GENERATE/FETCH API KEYS ---
    # The client automatically handles the session cookies set in Step 2.
    generate_keys_url = f"{ERPNEXT_URL}/api/method/frappe.core.doctype.user.user.generate_keys"
    
    print("Step 2: Requesting User API Keys from ERPNext...")
    keys_response = client.post(generate_keys_url)
    
    if keys_response.status_code == 200:
        keys_data = keys_response.json().get("message", {})
        api_key = keys_data.get("api_key")
        api_secret = keys_data.get("api_secret")
        
        print("🎉 SUCCESS! API Credentials Received:")
        print(f"   🔹 API Key:    {api_key}")
        print(f"   🔹 API Secret: {api_secret}")
        print("\n💡 You can now use these stateless tokens in your FastAPI bridge!")
    else:
        print("⚠️ Direct API generation endpoint skipped or not allowed. Trying fallback...")
        
        # FALLBACK: Try pulling standard logged-in user profile info
        fallback_url = f"{ERPNEXT_URL}/api/method/frappe.auth.get_logged_user"
        fallback_response = client.get(fallback_url)
        
        print(f"Status Code: {fallback_response.status_code}")
        print(f"Profile Check Response: {fallback_response.text}")


In [ ]:
import os
import json
import time
import logging
import threading
import requests
from typing import Dict, Any

logger = logging.getLogger("agent_system.auth")

class ERPNextAuthorizationCodeManager:
    """
    Manages OAuth2 Authorization Code flow tokens for ERPNext.
    Uses long-lived Refresh Tokens to fetch fresh Access Tokens completely in the background,
    avoiding interactive browser permission popups after initial setup.
    """
    def __init__(self, token_url: str, client_id: str, client_secret: str, scope: str = "all openid", cache_file: str = "tokens2.json"):
        self.token_url = token_url
        self.client_id = client_id
        self.client_secret = client_secret
        self.scope = scope
        self.cache_file = cache_file
        
        self._lock = threading.Lock()
        
        # Core runtime credentials state
        self._access_token: str | None = None
        self._refresh_token: str | None = None
        self._expires_at: float = 0.0
        self.expiry_buffer = 300  # 5-minute proactive refresh window

        # Load tokens from storage if they exist from a previous run
        self._load_token_cache()

    def _load_token_cache(self) -> str:
        """Loads existing tokens from disk to maintain state across process restarts."""
        if os.path.exists(self.cache_file):
            try:
                with open(self.cache_file, "r") as f:
                    data = json.load(f)
                    self._access_token = data.get("access_token")
                    self._refresh_token = data.get("refresh_token")
                    self._expires_at = float(data.get("expires_at", 0.0))
                print(data, "data found")
                logger.info("Loaded active session state from disk cache.")
            except Exception as e:
                logger.warning(f"Could not read token cache file: {e}. Starting fresh.")

    def _save_token_cache(self) :
        """Saves active tokens securely to disk storage."""
        print("start")
        try:
            with open(self.cache_file, "w") as f:
                json.dump({
                    "access_token": self._access_token,
                    "refresh_token": self._refresh_token,
                    "expires_at": self._expires_at
                }, f)
        except Exception as e:
            logger.error(f"Failed to persist tokens to disk cache: {e}")

    def _refresh_access_token(self) -> None:
        """Uses the long-lived refresh_token to securely pull a fresh 1-hour access_token."""
        if not self._refresh_token:
            raise ValueError(
                "No Refresh Token found! You must perform the initial interactive browser authorization "
                "via Postman or a configuration endpoint once, then populate the tokens.json cache file."
            )

        # Preparing fields based on your configuration parameters
        payload = {
            "grant_type": "refresh_token",
            "refresh_token": self._refresh_token,
            "client_id": self.client_id,
            "client_secret": self.client_secret,
            "scope": self.scope
        }
        headers = {"Content-Type": "application/x-www-form-urlencoded"}

        logger.info("Proactively exchanging refresh token for a new access token...")
        try:
            response = requests.post(self.token_url, data=payload, headers=headers, timeout=10)
            response.raise_for_status()
            data = response.json()

            self._access_token = data["access_token"]
            # Update refresh token if the server rolls/rotates it
            if "refresh_token" in data:
                self._refresh_token = data["refresh_token"]

            expires_in = float(data.get("expires_in", 3600))
            self._expires_at = time.time() + expires_in

            # Sync to storage immediately
            self._save_token_cache()
            logger.info("Tokens successfully rotated and stored securely.")
            
        except requests.RequestException as e:
            logger.critical(f"Failed to refresh background access token: {e}")
            raise RuntimeError(f"ERPNext rejected token rotation payload: {e}") from e

    def get_valid_token(self) -> str:
        """Thread-safe gateway ensuring agents always receive an active access token."""
        current_time = time.time()
        
        if current_time >= (self._expires_at - self.expiry_buffer):
            with self._lock:
                if time.time() >= (self._expires_at - self.expiry_buffer):
                    self._refresh_access_token()
                    
        return self._access_token


In [ ]:
from requests.auth import AuthBase

class ERPNextBearerAuth(AuthBase) :

    def __init__(self, token_manager) :
        self.token_manager = token_manager

    def __call__(self, r) :

        print("Accepting the valid token and creating authorization payload")
        token = self.token_manager.get_valid_token()
        r.headers["Authorization"] = f"Bearer {token}"

        print(f"r--- value -- {r.headers["Authorization"]}")

        return r


In [ ]:
import requests
from typing import Dict, Any

class ERPNextBaseClient:
    """
    Base API Client wrapping connections to ERPNext with auto-refreshing sessions.
    """
    def __init__(self, base_url: str, token_manager: ERPNextAuthorizationCodeManager):
        self.base_url = base_url.rstrip('/')
        self.session = requests.Session()
        
        # Seamless integration: Attach the auto-refreshing authorization engine
        self.session.auth = ERPNextBearerAuth(token_manager)
        
        # Enforce standard headers for ERPNext REST architecture
        self.session.headers.update({
            "Accept": "application/json",
            "Content-Type": "application/json"
        })

    def request(self, method: str, endpoint: str, **kwargs) -> Dict[str, Any]:
        """
        Centralized request method with forced production timeouts and status handling.
        """
        url = f"{self.base_url}/{endpoint.lstrip('/')}"
        
        # Default production protection constraints
        if "timeout" not in kwargs:
            kwargs["timeout"] = 30  # Keep agent threads from hanging indefinitely
            
        try:
            response = self.session.request(method, url, **kwargs)
            response.raise_for_status()
            
            # ERPNext wraps standard REST data inside a 'data' key object
            return response.json()
        except requests.HTTPError as e:
            if response.status_code == 401:
                # This should rarely occur due to our proactive token buffer checks
                raise PermissionError("ERPNext rejected credentials with 401 Unauthorized.") from e
            raise RuntimeError(f"ERPNext API returned unexpected error {response.status_code}: {response.text}") from e
